# Step 05: papers → claims → the literature graph

Each paper is read section by section by its source's reader (`pubmed.py`; `clinicaltrials.py`, which reads a trial registration like a paper; `arxiv.py`), then through the **reading pipeline** `[literature] pipeline` names, a folder in `src/pipelines/`. The biomedical one (`biomed/`) runs scispaCy (sentences, parse, entities, abbreviations), links mentions to concept IDs with the paper as context (`common/linking.py`), and turns every pair of concepts in a sentence into a **claim** (`common/relations.py`, with the `[lexicon]` of `biomed/defaults.toml`): a relation from the cue words between them, negated or hedged, from which section, and the paper's study type, species and date (`biomed/context.py`). Another domain, or your own model, is another folder there. Papers are cached in the pool, so each is read once per linker + pipeline settings.

The experiment's claims become its **literature graph**: one edge per (subject, relation, object), with how many papers state it, negate it, hedge it, report it as a finding, and when it was first seen. Step 02 merges it into the next round's **experiment graph**.

This notebook uses `experiments/examples.toml` and carries on from the 04 example (two rounds downloaded into `corpora/examples`). The first run reads the 16 papers (about 1.5 minutes on a CPU); after that they're cached.

In [1]:
import json
import os
import shutil
import stat
import subprocess
import sys
import tomllib
from pathlib import Path

import duckdb

SRC = Path.cwd().parent
LACUNA = SRC.parent
EXPERIMENT = LACUNA / 'experiments' / 'examples.toml'
experiment = tomllib.loads(EXPERIMENT.read_text())
runs, pool = LACUNA / 'runs' / 'examples', LACUNA / 'corpora' / 'examples'

def remove(path):
    """Delete a file or folder, clearing the read-only flag OneDrive puts on synced files."""
    def writable(func, p, _):
        os.chmod(p, stat.S_IWRITE)
        func(p)
    if path.is_dir():
        shutil.rmtree(path, onexc=writable)
    else:
        os.chmod(path, stat.S_IWRITE)
        path.unlink()


# Start where the 04 example ends: two rounds of topics, downloaded.
for path in [*runs.glob('02-expand/graph_*'), *runs.glob('03-topics/round_*.json'), *runs.glob('05-literature/round_*')]:
    if int(path.stem.split('_')[1]) > 2:
        remove(path)


def step(name, *extra):
    """Run one step's CLI on the examples experiment and show what it printed."""
    result = subprocess.run([sys.executable, SRC / name / 'run.py', '--experiment', EXPERIMENT, *extra],
                            capture_output=True, encoding='utf-8', errors='replace', env={**__import__('os').environ, 'PYTHONUTF8': '1'})
    print(result.stdout if result.returncode == 0 else result.stderr[-3000:])


step('05-literature')
cache = max((p for p in (pool / 'processed').iterdir() if json.loads((p / 'settings.json').read_text()).get('reader') == 'pubmed'),
            key=lambda p: (p / 'settings.json').stat().st_mtime)
graph = runs / '05-literature' / 'round_2'

examples clinicaltrials: 7 papers, 0 to read, this task 0, pipeline biomed -> C:\Users\harri\OneDrive - Indiana University\B659\pubmed\lacuna\corpora\examples\processed\5cd914b64f62
examples pubmed: 16 papers, 0 to read, this task 0, pipeline biomed -> C:\Users\harri\OneDrive - Indiana University\B659\pubmed\lacuna\corpora\examples\processed\fa10d85773d8
wrote C:\Users\harri\OneDrive - Indiana University\B659\pubmed\lacuna\runs\examples\05-literature\round_2 (1,035 nodes, 5,362 edges from 23 papers)



## 1. Reading a paper

Title and abstract from the PubMed record (a structured abstract labels its parts), the body from PMC by section. Back matter (references, acknowledgments, funding, conflicts) and citation markers are left out. A reader takes an entry of 04's `papers_<source>.json`; its paper ID (`pubmed:<pmid>`) is what the cache and the graph use.

In [2]:
sys.path[:0] = [str(SRC / '05-literature'), str(SRC / 'pipelines'), str(SRC / 'shared')]
import pubmed
import settings

# Every setting comes from a defaults file next to the code, with the experiment's overrides on top.
reading = settings.merged(SRC / '05-literature' / 'pubmed.toml', experiment['literature'].get('reader', {}).get('pubmed', {}))
entry = next(p for p in json.loads((runs / '04-download' / 'papers_pubmed.json').read_text()) if p['pmcid'])  # the first with full text
print(entry)
for p in pubmed.read(entry, pool, reading)[:9]:
    print(f"{p['section']:11} {str(p['label'] or ''):11} {p['text'][:95]}")

{'paper': 'pubmed:37322433', 'source': 'pubmed', 'pmid': '37322433', 'pmcid': 'PMC10268375', 'topics': ['glioblastoma']}
title                   Molecular mechanisms of tumor resistance to radiotherapy.
abstract    background  Cancer is the most prevalent cause of death globally, and radiotherapy is considered the standa
abstract    main body   In this review, we have extensively discussed several crucial aspects that cause resistance of 
abstract    conclusions Studying the molecular mechanisms responsible for radiotherapy resistance and its interactions 
intro                   Cancer incidence and mortality are increasing globally, making it one of the greatest challenge
intro                   In this review, we aim to discuss the mechanisms currently associated with radioresistance and 
other                   Genotoxic cancer therapy inactivates and kills cancer cells via extensive DNA damage, and RT is
other                   Tumor cells engage in classical NHEJ for repair prima

## 2. Reading a trial

A ClinicalTrials.gov study is read like a paper: its title, summary, description and outcomes (eligibility criteria are off by default: exclusion lists name diseases the trial isn't about). It also gives **facts**: every intervention × condition pair the registry coded with MeSH becomes a `tested_for` claim, labelled with the trial's status. A trial testing a drug isn't evidence that it works, so it's its own relation, not `treats`. Each source has its own cache.

In [3]:
import clinicaltrials

trial_reading = settings.merged(SRC / '05-literature' / 'clinicaltrials.toml', experiment['literature'].get('reader', {}).get('clinicaltrials', {}))
trials = json.loads((runs / '04-download' / 'papers_clinicaltrials.json').read_text())
trial = next(t for t in trials if clinicaltrials.facts(t, pool, trial_reading))  # the first with coded interventions
for p in clinicaltrials.read(trial, pool, trial_reading)[:5]:
    print(f"{p['section']:12} {p['text'][:100]}")
for f in clinicaltrials.facts(trial, pool, trial_reading):
    print(f"fact: {f['s_text']} -{f['r']}-> {f['o_text']}   ({f['quote']})")

trials_cache = max((p for p in (pool / 'processed').iterdir() if json.loads((p / 'settings.json').read_text()).get('reader') == 'clinicaltrials'),
                   key=lambda p: (p / 'settings.json').stat().st_mtime)
print(duckdb.sql(f"SELECT section, count(*) AS claims, count(DISTINCT paper) AS trials "
                 f"FROM read_parquet('{(trials_cache / 'claims').as_posix()}/*.parquet') GROUP BY ALL ORDER BY claims DESC"))

title        A Phase I Trial of High-Dose Ascorbate in Glioblastoma Multiforme
summary      This is a phase 1 (first in man) study testing the safety of adding high dose ascorbate (vitamin C) 
description  This phase 1 study will test the safety of adding high dose ascorbate (vitamin C) to standard chemor
description  Standard treatment for glioblastoma multiforme (GBM) involves surgery followed by radiation combined
description  Participants will:
fact: Ascorbic Acid -tested_for-> Glioblastoma   (NCT01752491: PHASE1, completed)
fact: Temozolomide -tested_for-> Glioblastoma   (NCT01752491: PHASE1, completed)
fact: Radiotherapy -tested_for-> Glioblastoma   (NCT01752491: PHASE1, completed)
┌─────────────┬────────┬────────┐
│   section   │ claims │ trials │
│   varchar   │ int64  │ int64  │
├─────────────┼────────┼────────┤
│ description │    117 │      6 │
│ title       │     27 │      6 │
│ outcomes    │     26 │      4 │
│ summary     │     25 │      7 │
│ registry    │     24 │      6

## 3. The pipeline on a few sentences

The same call step 05 makes: `build(experiment, root, device, settings)` returns a pipeline, and `read()` takes papers (here one made-up abstract) and gives back their sentences, mentions and claims. "TMZ" and "GBM" are linked through the abbreviations the text defines; "did not prolong" makes a negated *treats* claim; "may" hedges; "including but not limited to" is a pseudo-negation; and coreference links "It" and "The drug" to the concepts they stand for. (`co_occurs` claims are left out here.)

In [4]:
import importlib

name = experiment['literature']['pipeline']
piping = settings.merged(SRC / 'pipelines' / name / 'defaults.toml', experiment['literature'].get(name, {}))
pipeline = importlib.import_module(f'{name}.pipeline').build(experiment, LACUNA / 'data', 'cpu', piping)

text = ('Temozolomide (TMZ) did not prolong survival in patients with recurrent glioblastoma (GBM). '
        'TMZ may inhibit MGMT expression in GBM cells. '
        'IDH1 mutations are associated with longer survival in glioma, including but not limited to astrocytoma. '
        'Bevacizumab was given to 40 patients. It reduced tumour oedema in glioblastoma. '
        'The drug did not improve overall survival in glioblastoma.')
paper = ('example', [{'section': 'abstract', 'label': None, 'text': text}], {'date': None, 'publication_types': [], 'mesh': []})
sentences, mentions, claims = pipeline.read([paper])[0]
for s in sentences:
    print(s['text'])
    print('   mentions:', [(m['text'], m['id'], m['method']) for m in mentions if m['i'] == s['i']])
    for c in claims:
        if c['i'] == s['i'] and c['r'] != 'co_occurs':
            print(f"   claim: {c['s_text']} -{c['r']}-> {c['o_text']}   negated={c['negated']} ({c['negation_cue']}) "
                  f"hedged={c['hedged']} ({c['hedge_cue']})")

C:\Users\harri\OneDrive - Indiana University\B659\pubmed\lacuna\.env\Lib\site-packages\cupy\_environment.py:215: UserWarning: CUDA path could not be detected. Set CUDA_PATH environment variable if CuPy fails to load.
  warnings.warn(


C:\Users\harri\OneDrive - Indiana University\B659\pubmed\lacuna\.env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\harri\OneDrive - Indiana University\B659\pubmed\lacuna\.env\Lib\site-packages\spacy\language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


10/29/2026 18:40:46 - INFO - 	 missing_keys: []


10/29/2026 18:40:46 - INFO - 	 unexpected_keys: []


...


10/29/2026 18:40:58 - INFO - 	 ***** Running Inference on 1 texts *****


Inference:   0%|          | 0/1 [00:00<?, ?it/s]

Inference: 100%|██████████| 1/1 [00:00<00:00,  1.67it/s]

Inference: 100%|██████████| 1/1 [00:00<00:00,  1.67it/s]

Temozolomide (TMZ) did not prolong survival in patients with recurrent glioblastoma (GBM).
   mentions: [('Temozolomide', 'mesh:D000077204', 'exact'), ('TMZ', 'mesh:D000077204', 'abbreviation'), ('glioblastoma', 'mesh:D005909', 'exact'), ('GBM', 'mesh:D005909', 'abbreviation')]
   claim: Temozolomide -treats-> glioblastoma   negated=True (not) hedged=False (None)
TMZ may inhibit MGMT expression in GBM cells.
   mentions: [('TMZ', 'mesh:D000077204', 'abbreviation'), ('MGMT', 'ncbigene:4255', 'exact'), ('GBM', 'mesh:D005909', 'abbreviation')]
   claim: TMZ -inhibits-> MGMT   negated=False (None) hedged=True (may)
   claim: TMZ -inhibits-> GBM   negated=False (None) hedged=True (may)
IDH1 mutations are associated with longer survival in glioma, including but not limited to astrocytoma.
   mentions: [('IDH1', 'ncbigene:3417', 'exact'), ('glioma', 'mesh:D005910', 'exact'), ('astrocytoma', 'mesh:D001254', 'exact')]
   claim: glioma -associated_with-> IDH1   negated=False (None) hedged=False 

## 4. What the examples papers said

Claims by relation, a few typed claims from the papers' findings (Results, Conclusions, abstracts), and the literature graph's best-supported typed edges.

In [5]:
claims = f"read_parquet('{(cache / 'claims').as_posix()}/*.parquet')"
print(duckdb.sql(f'SELECT r, count(*) AS claims, sum(negated::INT) AS negated, sum(hedged::INT) AS hedged '
                 f'FROM {claims} GROUP BY r ORDER BY claims DESC'))
print(duckdb.sql(f"SELECT s_text, r, o_text, negated, hedged, section, left(quote, 70) AS quote FROM {claims} "
                 f"WHERE r IN ('treats', 'inhibits', 'activates') AND \"primary\" ORDER BY paper DESC, i LIMIT 10"))
print(duckdb.sql(f"SELECT e.r, s.name AS subject, o.name AS object, e.n_papers, e.n_negated, e.n_hedged, e.first_seen "
                 f"FROM '{(graph / 'edges.parquet').as_posix()}' e "
                 f"JOIN '{(graph / 'nodes.parquet').as_posix()}' s ON s.id = e.s JOIN '{(graph / 'nodes.parquet').as_posix()}' o ON o.id = e.o "
                 f"WHERE e.r <> 'co_occurs' ORDER BY e.n_papers DESC LIMIT 10"))
print(duckdb.sql(f"SELECT s.name AS intervention, o.name AS condition, e.n_papers AS trials "
                 f"FROM '{(graph / 'edges.parquet').as_posix()}' e "
                 f"JOIN '{(graph / 'nodes.parquet').as_posix()}' s ON s.id = e.s JOIN '{(graph / 'nodes.parquet').as_posix()}' o ON o.id = e.o "
                 f"WHERE e.r = 'tested_for' ORDER BY e.n_papers DESC LIMIT 8"))

┌─────────────────┬────────┬─────────┬────────┐
│        r        │ claims │ negated │ hedged │
│     varchar     │ int64  │ int128  │ int128 │
├─────────────────┼────────┼─────────┼────────┤
│ co_occurs       │   5228 │     194 │    491 │
│ associated_with │    535 │      31 │     88 │
│ activates       │    519 │      13 │     86 │
│ inhibits        │    488 │      12 │     68 │
│ causes          │    303 │       8 │     22 │
│ treats          │    185 │       8 │     34 │
└─────────────────┴────────┴─────────┴────────┘



┌─────────────────────┬───────────┬───────────────┬─────────┬─────────┬─────────────┬────────────────────────────────────────────────────────────────────────┐
│       s_text        │     r     │    o_text     │ negated │ hedged  │   section   │                                 quote                                  │
│       varchar       │  varchar  │    varchar    │ boolean │ boolean │   varchar   │                                varchar                                 │
├─────────────────────┼───────────┼───────────────┼─────────┼─────────┼─────────────┼────────────────────────────────────────────────────────────────────────┤
│ IDH                 │ inhibits  │ RT            │ false   │ false   │ conclusions │ For grade 3 tumors and grade 2 tumors unsuitable for observation/voras │
│ IDH                 │ inhibits  │ chemotherapy  │ false   │ false   │ conclusions │ For grade 3 tumors and grade 2 tumors unsuitable for observation/voras │
│ immune system       │ activates │ DIPG     

## 5. Closing the loop

Step 02 merges the literature graph into the next experiment graph: a concept the papers connect to it gets distance = its nearest neighbour's + 1 and reason `literature`, and every node a `support` (papers backing its literature edges). Step 03 ranks by it, so round 3 picks what the papers keep discussing with glioblastoma, not just what MeSH puts next to it.

In [6]:
step('02-expand')
print(duckdb.sql(f"SELECT name, types, distance, reason, support FROM '{(runs / '02-expand' / 'graph_3' / 'nodes.parquet').as_posix()}' "
                 f"WHERE distance <= 1 ORDER BY distance, support DESC LIMIT 15"))
step('03-topics')

merged C:\Users\harri\OneDrive - Indiana University\B659\pubmed\lacuna\runs\examples\05-literature\round_2
wrote C:\Users\harri\OneDrive - Indiana University\B659\pubmed\lacuna\runs\examples\02-expand\graph_3 (133 nodes, 5,378 edges from 1 seeds)

┌──────────────────────────────────┬─────────────┬──────────┬────────────┬─────────┐
│               name               │    types    │ distance │   reason   │ support │
│             varchar              │  varchar[]  │  int64   │  varchar   │  int64  │
├──────────────────────────────────┼─────────────┼──────────┼────────────┼─────────┤
│ Glioblastoma                     │ [disease]   │        0 │ seed       │     284 │
│ Glioma                           │ [disease]   │        1 │ ancestor   │     367 │
│ Diffuse Intrinsic Pontine Glioma │ [disease]   │        1 │ relative   │     269 │
│ Radiation                        │ [process]   │        1 │ literature │     171 │
│ Astrocytoma                      │ [disease]   │        1 │ ancestor  

oligodendroglioma                        pubmed    2,715 ->     3  clinicaltrials      422 ->     3
medulloblastoma                          pubmed    6,200 ->     3  clinicaltrials      348 ->     3
isocitrate_dehydrogenase                 pubmed    6,480 ->     3  clinicaltrials       48 ->     3
wrote C:\Users\harri\OneDrive - Indiana University\B659\pubmed\lacuna\runs\examples\03-topics\round_3.json (3 topics, 25 of 20,000 budget papers used across rounds)



## From the command line

```
python src/05-literature/run.py --experiment experiments/default.toml                    # read, then build the graph
python src/05-literature/run.py --experiment experiments/default.toml --device gpu
sbatch bin/literature.slurm experiments/default.toml                                     # on Quartz: GPU array
python src/05-literature/run.py --experiment experiments/default.toml --graph-only       # after the array
```
The caches, one per source: `corpora/<pool>/processed/<fingerprint>/` (`sentences`, `mentions`, `claims`, `papers`; `settings.json` says which source). The literature graph: `runs/<experiment>/05-literature/round_<n>/` (`nodes.parquet`, `edges.parquet`).